## Init

Muchas de las funciones que uso las importo directamente de funciones_tesis.py, que está en:\
https://github.com/PedroRozin/Tesis2025/blob/main/python/funciones_tesis.py




In [4]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import joblib
import funciones_tesis as ft
from funciones_tesis import RegressionNN, ImprovedRegressionNN 

## Cosas:

La red está entrenada con esta grilla:\
https://github.com/PedroRozin/Tesis2025/tree/main/outputs_pedro/grillas/params_para_entrenamiento_v1/grilla_results_para_entrenamiento.csv
\
\
La red entrenada buena está en esta carpeta:\
https://github.com/PedroRozin/Tesis2025/tree/main/outputs_pedro/neural_networks/tanh_buena_v2
\
\
El código con el que se entrenó está en:\
https://github.com/PedroRozin/Tesis2025/blob/main/python/grilla_NN.py

## Condiciones iniciales a partir de la red
Hay que descargarse la carpeta entera y cambiar los paths que sean necesarios (en teoría, solo se debería cambiar 'root_path' al path donde tengan descargada la carpeta 'tanh_buena_v2').

### Cargar modelo y scalers

In [ ]:
"""cargar modelo entrenado y los scalers"""

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') #usar GPU si está disponible
print(f"Using device: {device}")

model = ImprovedRegressionNN(activation='tanh')
folder_path = 'tanh_buena_v2'
network_name = f'_{folder_path}'

#cambiar paths a donde te descargues la carpeta
root_path = f'/home/pedrorozin/scripts/outputs_pedro/neural_networks/{folder_path}'  #este es mi path base hasta la carpeta donde tengo las redes
path_model = f'{root_path}/regression_model{network_name}.pth'
path_training_history = f'{root_path}/training_history{network_name}.csv' #opcional
path_final_metrics = f'{root_path}/final_metrics{network_name}.csv' #opcional
path_scaler_X = f'{root_path}/scaler_X{network_name}.pkl'
path_scaler_y = f'{root_path}/scaler_y{network_name}.pkl'


# cargar modelo entrenado y los scalers
model.load_state_dict(torch.load(path_model, map_location=device))  
model.eval()
scaler_X = joblib.load(path_scaler_X)
scaler_y = joblib.load(path_scaler_y)

features = ["a", "k h", "h", "Omega_m"] #la red está entrenada con estos features


Using device: cuda


### Predicción de UN ÚNICO PAR de condiciones iniciales dado un ÚNICO conjunto de parámetros.

In [13]:
"""obtengo una predicción para un único conjunto de parámetros {a, kh, h, Omega_m}"""

parametros_random = [0.03, 0.01, 0.68, .3] #a_ini, kh (h/Mpc), h, Omega_m
#manera muy ineficiente de hacer la predicción:
a_ini, kh, h, Omega_m = parametros_random
X_single = np.array([[a_ini, kh, h, Omega_m]])

# escalar con los scalers que importamos
X_single_scaled = scaler_X.transform(X_single)
X_single_tensor = torch.tensor(X_single_scaled, dtype=torch.float32)

# evaluar en la red
with torch.no_grad():
    y_pred_scaled = model(X_single_tensor).numpy()

# desescalar
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# ver los de delta_m y delta_prime_m
delta_m_pred, delta_prime_m_pred = y_pred[0, 0], y_pred[0, 1]
print(f"delta_m_pred: {delta_m_pred}")
print(f"delta_prime_m_pred: {delta_prime_m_pred}")

delta_m_pred: -29.197172164916992
delta_prime_m_pred: -936.53125


### Predicción de varios puntos y precision de la red
A la red le podemos pasar varios conjuntos de parámetros {$a$, $kh$, $h$, $\Omega_m$} para que prediga $\delta_m$ y $\delta'_m$. En particular, le podemos pasar todos los puntos de la grilla con la que entrenamos la red.\
\
**PARA CORRER ESTO HAY QUE TENER DESCARGADA LA GRILLA**

In [22]:
path_data = f'/home/pedrorozin/scripts/outputs_pedro/grillas/params_para_entrenamiento_v1/grilla_results_para_entrenamiento.csv' #datos de entrenamiento

df_path = pd.read_csv(path_data)
mask = (
	(df_path['k h'] <= 0.2) &
	(df_path['a'] < 0.0345) &
	(df_path['h'] > 0.642) &
	(df_path['h'] < 0.76) &
	(df_path['Omega_m'] > 0.154) &
	(df_path['Omega_m'] < 0.444)
) #saco los bordes donde no quedó bien entrenada la red


for idx, row in df_path.iterrows(): #mascara para evitar termino de borde en k h = k horizon
	k_h = row['k h']
	a = row['a']
	k_hor = row['k_horizon']
	if k_h < k_hor*(1 + 0.2):
		mask.loc[idx] = False


df = df_path[mask].copy()

features = ["a", "k h", "h", "Omega_m"]
X = df[features].values
targets = ["delta_m", "delta_prime_m"]
y = df[targets].values

# escalar
X_scaled = scaler_X.transform(X)
y_scaled = scaler_y.transform(y)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)
# evaluar en la red
with torch.no_grad():
	y_pred_scaled = model(X_tensor).numpy()
# desescalar
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# create DataFrame con los resultados físicos (sin escalar)
results = pd.DataFrame(y_pred, columns=["delta_m", "delta_prime_m"])

print("=" * 50)
print("Primeras 5 filas - Red Neuronal (desescalada):")
display(results.head(10))
print("\nPrimeras 5 filas - Grilla (valores posta):")  
display(df[targets].head(10))

Primeras 5 filas - Red Neuronal (desescalada):


,delta_m,delta_prime_m
0,-328.438293,-10781.424805
1,-406.465057,-13331.020508
2,-76.138458,-2477.419922
3,-21.129576,-666.985352
4,-57.689026,-1870.511719
5,-30.127287,-960.256836
6,-42.413727,-1359.494141
7,-326.363831,-10613.954102
8,-96.023438,-3112.524414
9,-408.254639,-13272.103516



Primeras 5 filas - Grilla (valores posta):


,delta_m,delta_prime_m
16018,-328.488271,-10783.630016
16019,-406.411049,-13330.721370
16021,-76.100558,-2476.419689
16022,-21.096538,-664.325912
16023,-57.705686,-1870.129058
16024,-30.134425,-960.495119
16030,-42.413681,-1359.182482
16031,-326.434695,-10618.052171
16032,-96.016554,-3110.579993
16033,-408.267505,-13269.974948


Acá vemos que los resultados finales de la red vs grilla

In [24]:
from IPython.display import display 

# ver los de delta_m y delta_prime_m
delta_m_pred_all = y_pred[:, 0]
delta_prime_m_pred_all = y_pred[:, 1]
# comparar con los reales
y_pred_fisicos = pd.DataFrame(y_pred, columns=targets)
y_fisicos = pd.DataFrame(y, columns=targets)
comparison_df = pd.concat([y_fisicos, y_pred_fisicos], axis=1, keys=['Real', 'Predicted'])

df_merged = pd.concat([df.reset_index(drop=True), y_fisicos, y_pred_fisicos], axis=1)

# Función para calcular la diferencia porcentual
diferencia = lambda x, y: np.abs(x - y) / np.abs(y) * 100

# 1. Crear un DataFrame específico para mostrar los resultados combinando features y errores
df_resultados = df.reset_index(drop=True)[features].copy()

# Calcular las diferencias y guardarlas como nuevas columnas
for target in targets:
    col_name = f'diff_{target} (%)'
    df_resultados[col_name] = diferencia(y_pred_fisicos[target], y_fisicos[target])

# 2. Obtener el Top 100 de mayores diferencias en delta_m
# Ordenamos de forma descendente y tomamos los primeros 100
top_100_diff = df_resultados.sort_values(by='diff_delta_m (%)', ascending=False).head(100)

print("Top 100 diferencias porcentuales más altas en delta_m:")
display(top_100_diff)

# 3. Generar la tabla de estadísticas descriptivas
# Seleccionamos solo las columnas de diferencias y usamos el método .describe() de pandas
columnas_diff = [f'diff_{t} (%)' for t in targets]
stats_df = df_resultados[columnas_diff].describe()

print("\nEstadísticas descriptivas de las diferencias porcentuales:")
display(stats_df)

Top 100 diferencias porcentuales más altas en delta_m:


,a,k h,h,Omega_m,diff_delta_m (%),diff_delta_prime_m (%)
102340,0.030600,0.007311,0.696,0.226,0.510223,0.512477
44009,0.030529,0.006741,0.758,0.184,0.499821,0.615037
99391,0.030583,0.007287,0.690,0.224,0.496384,0.479511
92604,0.030055,0.007196,0.758,0.218,0.490751,0.430650
67039,0.030063,0.006960,0.758,0.200,0.487863,0.509839
...,...,...,...,...,...,...
178876,0.030541,0.007974,0.684,0.282,0.317418,0.367089
171325,0.030590,0.007902,0.708,0.276,0.316484,0.342018
94010,0.030498,0.007234,0.700,0.220,0.316214,0.262555
107988,0.030524,0.007360,0.698,0.230,0.313515,0.284561



Estadísticas descriptivas de las diferencias porcentuales:


,diff_delta_m (%),diff_delta_prime_m (%)
count,3.848750e+05,3.848750e+05
mean,9.493692e-03,9.845769e-03
std,1.977945e-02,2.126084e-02
min,2.343467e-09,1.483840e-08
25%,2.067518e-03,2.137552e-03
50%,4.543014e-03,4.668216e-03
75%,8.833985e-03,9.105030e-03
max,5.102229e-01,6.150374e-01


## Integración con las condiciones iniciales
Hay varios códigos en este github que dan la solución numérica a las perturbaciones con las condiciones dadas por la red. Uno de estos está en https://github.com/PedroRozin/Tesis2025/blob/main/notebooks/results_NN.ipynb (acá también se ve la precisión que tiene la red).
\
\
El código más completo y eficiente para computar soluciones es https://github.com/PedroRozin/Tesis2025/blob/main/notebooks/fsigma8_MG.ipynb 